In [2]:
import datasets
import pandas as pd
import numpy as np
import json
import huggingface_hub
from tqdm import tqdm
import os

In [30]:
def get_json(file):
  local = os.path.join(local_repo, file)
  obj = json.load(open(local, "r"))
  return obj

def list_files_recursive(path='.'):
    for entry in os.listdir(path):
        full_path = os.path.join(path, entry)
        if os.path.isdir(full_path):
            files = [*list_files_recursive(full_path)]
            for file in files:
                yield file
        else:
            yield full_path



# local_repo = huggingface_hub.snapshot_download(repo_id="canrager/graphing_eval_results_0122", repo_type="dataset")
local_repo = "C:\\Users\\alexg\\Documents\\code\\graphing_eval_results_0122"

In [40]:
# get a list of all files in the repo recursively
files = list(list_files_recursive(local_repo))
# remove the first 2 directories
dir_files = []
print(files)
suffix = "_eval_results.json"
metric_files = {
  "absorption": {os.path.split(file)[-1][:-len(suffix)]: file for file in files if "absorption" in file and suffix in file},
  "autointerp": {os.path.split(file)[-1][:-len(suffix)]: file for file in files if "autointerp" in file and suffix in file},
  "core": {os.path.split(file)[-1][:-len(suffix)]: file for file in files if "core" in file and suffix in file},
  "scr": {os.path.split(file)[-1][:-len(suffix)]: file for file in files if "scr" in file and suffix in file},
  "sparse_probing": {os.path.split(file)[-1][:-len(suffix)]: file for file in files if "sparse_prob" in file and suffix in file},
  "tpp": {os.path.split(file)[-1][:-len(suffix)]: file for file in files if "tpp" in file and suffix in file},
  "unlearning": {os.path.split(file)[-1][:-len(suffix)]: file for file in files if "unlearning" in file and suffix in file},
}

display(metric_files)

['C:\\Users\\alexg\\Documents\\code\\graphing_eval_results_0122\\.git\\config', 'C:\\Users\\alexg\\Documents\\code\\graphing_eval_results_0122\\.git\\description', 'C:\\Users\\alexg\\Documents\\code\\graphing_eval_results_0122\\.git\\HEAD', 'C:\\Users\\alexg\\Documents\\code\\graphing_eval_results_0122\\.git\\hooks\\applypatch-msg.sample', 'C:\\Users\\alexg\\Documents\\code\\graphing_eval_results_0122\\.git\\hooks\\commit-msg.sample', 'C:\\Users\\alexg\\Documents\\code\\graphing_eval_results_0122\\.git\\hooks\\fsmonitor-watchman.sample', 'C:\\Users\\alexg\\Documents\\code\\graphing_eval_results_0122\\.git\\hooks\\post-update.sample', 'C:\\Users\\alexg\\Documents\\code\\graphing_eval_results_0122\\.git\\hooks\\pre-applypatch.sample', 'C:\\Users\\alexg\\Documents\\code\\graphing_eval_results_0122\\.git\\hooks\\pre-commit.sample', 'C:\\Users\\alexg\\Documents\\code\\graphing_eval_results_0122\\.git\\hooks\\pre-merge-commit.sample', 'C:\\Users\\alexg\\Documents\\code\\graphing_eval_results

{'absorption': {'gemma-2-2b_layer_12_pca_sae_custom_sae': 'C:\\Users\\alexg\\Documents\\code\\graphing_eval_results_0122\\absorption\\gemma-2-2b_layer_12_pca_sae_custom_sae_eval_results.json',
  'saebench_gemma-2-2b_width-2pow12_date-0108_BatchTopK_gemma-2-2b__0108_resid_post_layer_12_trainer_0': 'C:\\Users\\alexg\\Documents\\code\\graphing_eval_results_0122\\absorption\\saebench_gemma-2-2b_width-2pow12_date-0108_BatchTopK_gemma-2-2b__0108_resid_post_layer_12_trainer_0_eval_results.json',
  'saebench_gemma-2-2b_width-2pow12_date-0108_BatchTopK_gemma-2-2b__0108_resid_post_layer_12_trainer_1': 'C:\\Users\\alexg\\Documents\\code\\graphing_eval_results_0122\\absorption\\saebench_gemma-2-2b_width-2pow12_date-0108_BatchTopK_gemma-2-2b__0108_resid_post_layer_12_trainer_1_eval_results.json',
  'saebench_gemma-2-2b_width-2pow12_date-0108_BatchTopK_gemma-2-2b__0108_resid_post_layer_12_trainer_2': 'C:\\Users\\alexg\\Documents\\code\\graphing_eval_results_0122\\absorption\\saebench_gemma-2-2b_widt

In [41]:
from collections import defaultdict


saes = defaultdict(dict)

In [42]:
# files = ["autointerp/saebench_gemma-2-2b_width-2pow12_date-0108/saebench_gemma-2-2b_width-2pow12_date-0108_BatchTopK_gemma-2-2b__0108_resid_post_layer_12_trainer_0_eval_results.json", "autointerp/saebench_gemma-2-2b_width-2pow12_date-0108/saebench_gemma-2-2b_width-2pow12_date-0108_BatchTopK_gemma-2-2b__0108_resid_post_layer_12_trainer_1_eval_results.json"]

for metric, files in metric_files.items():
    for sae, file in tqdm(files.items(), desc=metric):
        eval_result_metrics = get_json(file)["eval_result_metrics"]
        # print(f"eval_result_metrics: {eval_result_metrics}")
        saes[sae].update({metric: eval_result_metrics})
        # tqdm.write(f"{sae}: {eval_result_metrics}")

unlearning: 100%|██████████| 169/169 [00:01<00:00, 94.04it/s] 


In [44]:
with open("saes_metrics.json", "w") as f:
    json.dump(saes, f, indent=4)
    print(f"Saved saes metrics to saes_metrics.json")

Saved saes metrics to saes_metrics.json


In [ ]:
# unpack nested dicts in dataframe
saes_df = pd.json_normalize(saes, sep="/")
saes_df

,gemma-2-2b_layer_12_pca_sae_custom_sae/absorption/mean/mean_absorption_fraction_score,gemma-2-2b_layer_12_pca_sae_custom_sae/absorption/mean/mean_num_split_features,gemma-2-2b_layer_12_pca_sae_custom_sae/autointerp/autointerp/autointerp_score,gemma-2-2b_layer_12_pca_sae_custom_sae/core/model_behavior_preservation/kl_div_score,gemma-2-2b_layer_12_pca_sae_custom_sae/core/model_behavior_preservation/kl_div_with_ablation,gemma-2-2b_layer_12_pca_sae_custom_sae/core/model_behavior_preservation/kl_div_with_sae,gemma-2-2b_layer_12_pca_sae_custom_sae/core/model_performance_preservation/ce_loss_score,gemma-2-2b_layer_12_pca_sae_custom_sae/core/model_performance_preservation/ce_loss_with_ablation,gemma-2-2b_layer_12_pca_sae_custom_sae/core/model_performance_preservation/ce_loss_with_sae,gemma-2-2b_layer_12_pca_sae_custom_sae/core/model_performance_preservation/ce_loss_without_sae,...,saebench_pythia-160m-deduped_width-2pow14_date-0108_Standard_pythia-160m-deduped__0104_resid_post_layer_8_trainer_5/tpp/tpp_metrics/tpp_threshold_20_unintended_diff_only,saebench_pythia-160m-deduped_width-2pow14_date-0108_Standard_pythia-160m-deduped__0104_resid_post_layer_8_trainer_5/tpp/tpp_metrics/tpp_threshold_50_total_metric,saebench_pythia-160m-deduped_width-2pow14_date-0108_Standard_pythia-160m-deduped__0104_resid_post_layer_8_trainer_5/tpp/tpp_metrics/tpp_threshold_50_intended_diff_only,saebench_pythia-160m-deduped_width-2pow14_date-0108_Standard_pythia-160m-deduped__0104_resid_post_layer_8_trainer_5/tpp/tpp_metrics/tpp_threshold_50_unintended_diff_only,saebench_pythia-160m-deduped_width-2pow14_date-0108_Standard_pythia-160m-deduped__0104_resid_post_layer_8_trainer_5/tpp/tpp_metrics/tpp_threshold_100_total_metric,saebench_pythia-160m-deduped_width-2pow14_date-0108_Standard_pythia-160m-deduped__0104_resid_post_layer_8_trainer_5/tpp/tpp_metrics/tpp_threshold_100_intended_diff_only,saebench_pythia-160m-deduped_width-2pow14_date-0108_Standard_pythia-160m-deduped__0104_resid_post_layer_8_trainer_5/tpp/tpp_metrics/tpp_threshold_100_unintended_diff_only,saebench_pythia-160m-deduped_width-2pow14_date-0108_Standard_pythia-160m-deduped__0104_resid_post_layer_8_trainer_5/tpp/tpp_metrics/tpp_threshold_500_total_metric,saebench_pythia-160m-deduped_width-2pow14_date-0108_Standard_pythia-160m-deduped__0104_resid_post_layer_8_trainer_5/tpp/tpp_metrics/tpp_threshold_500_intended_diff_only,saebench_pythia-160m-deduped_width-2pow14_date-0108_Standard_pythia-160m-deduped__0104_resid_post_layer_8_trainer_5/tpp/tpp_metrics/tpp_threshold_500_unintended_diff_only
0,0.005806,1.076923,0.670643,0.999937,10.0625,0.000633,1.0,12.4375,2.9375,2.9375,...,0.003275,0.04025,0.0437,0.00345,0.0639,0.0707,0.0068,0.18285,0.1953,0.01245


In [87]:
for sae, file in tqdm(core_files.items()):
    local = huggingface_hub.hf_hub_download(repo_id="canrager/graphing_eval_results_0122", filename=file, repo_type="dataset")
    obj = json.load(open(local, "r"))
    # print(repo.to_dict())
    eval_result_metrics = obj["eval_result_metrics"]
    # print(f"eval_result_metrics: {eval_result_metrics}")
    saes[sae].update({"core": eval_result_metrics})
    # tqdm.write(f"{sae}: {eval_result_metrics}")

  1%|▏         | 4/277 [00:00<00:15, 18.20it/s]

gemma-2-2b_layer_12_pca_sae_custom_sae: {'model_behavior_preservation': {'kl_div_score': 0.9999370693420031, 'kl_div_with_ablation': 10.0625, 'kl_div_with_sae': 0.00063323974609375}, 'model_performance_preservation': {'ce_loss_score': 1.0, 'ce_loss_with_ablation': 12.4375, 'ce_loss_with_sae': 2.9375, 'ce_loss_without_sae': 2.9375}, 'reconstruction_quality': {'explained_variance': 1.0, 'mse': 8.296966552734375e-05, 'cossim': 1.0}, 'shrinkage': {'l2_norm_in': 149.0, 'l2_norm_out': 149.0, 'l2_ratio': 1.0, 'relative_reconstruction_bias': 1.0}, 'sparsity': {'l0': 2304.0, 'l1': -27.375}, 'token_stats': {'total_tokens_eval_reconstruction': 409600, 'total_tokens_eval_sparsity_variance': 4096000}}
saebench_gemma-2-2b_width-2pow12_date-0108_BatchTopK_gemma-2-2b__0108_resid_post_layer_12_trainer_0: {'model_behavior_preservation': {'kl_div_score': 0.9505046583850931, 'kl_div_with_ablation': 10.0625, 'kl_div_with_sae': 0.498046875}, 'model_performance_preservation': {'ce_loss_score': 0.949013157894

  2%|▏         | 6/277 [00:00<00:14, 18.24it/s]

saebench_gemma-2-2b_width-2pow12_date-0108_BatchTopK_gemma-2-2b__0108_resid_post_layer_12_trainer_3: {'model_behavior_preservation': {'kl_div_score': 0.9888392857142857, 'kl_div_with_ablation': 10.0625, 'kl_div_with_sae': 0.1123046875}, 'model_performance_preservation': {'ce_loss_score': 0.9884868421052632, 'ce_loss_with_ablation': 12.4375, 'ce_loss_with_sae': 3.046875, 'ce_loss_without_sae': 2.9375}, 'reconstruction_quality': {'explained_variance': 0.73046875, 'mse': 1.65625, 'cossim': 0.91796875}, 'shrinkage': {'l2_norm_in': 149.0, 'l2_norm_out': 137.0, 'l2_ratio': 0.91796875, 'relative_reconstruction_bias': 1.0}, 'sparsity': {'l0': 164.01565551757812, 'l1': 892.0}, 'token_stats': {'total_tokens_eval_reconstruction': 409600, 'total_tokens_eval_sparsity_variance': 4096000}, 'misc_metrics': {'freq_over_1_percent': 0.676025390625, 'freq_over_10_percent': 0.07275390625, 'normalized_freq_over_1_percent': 0.965038537979126, 'normalized_freq_over_10_percent': 0.279417484998703, 'average_max

  4%|▍         | 11/277 [00:00<00:14, 18.72it/s]

...

 76%|███████▌  | 211/277 [00:27<00:09,  6.86it/s]

saebench_pythia-160m-deduped_width-2pow14_date-0108_BatchTopK_pythia-160m-deduped__0108_resid_post_layer_8_trainer_5: {'model_behavior_preservation': {'kl_div_score': -1.0, 'kl_div_with_ablation': -1.0, 'kl_div_with_sae': -1.0}, 'model_performance_preservation': {'ce_loss_score': 0.9990740653938155, 'ce_loss_with_ablation': 11.921521186828613, 'ce_loss_with_sae': 4.197295665740967, 'ce_loss_without_sae': 4.190136909484863}, 'reconstruction_quality': {'explained_variance': 0.9882590770721436, 'mse': 0.0013835941208526492, 'cossim': 0.9977225661277771}, 'shrinkage': {'l2_norm_in': 26.8211669921875, 'l2_norm_out': 26.75553321838379, 'l2_ratio': 0.9971595406532288, 'relative_reconstruction_bias': 1.000812292098999}, 'sparsity': {'l0': 629.2872924804688, 'l1': 592.595458984375}, 'token_stats': {'total_tokens_eval_reconstruction': 409600, 'total_tokens_eval_sparsity_variance': 4096000}, 'misc_metrics': {'freq_over_1_percent': 0.0472412109375, 'freq_over_10_percent': 0.047119140625, 'normaliz

(…)post_layer_8_trainer_0_eval_results.json:   0%|          | 0.00/3.51k [00:00<?, ?B/s]

 76%|███████▌  | 211/277 [00:27<00:08,  7.68it/s]


KeyError: 'saebench_pythia-160m-deduped_width-2pow14_date-0108_GatedSAE_pythia-160m-deduped__0108_resid_post_layer_8_trainer_0'

In [ ]:
for sae, file in tqdm(core_files.items()):
    local = huggingface_hub.hf_hub_download(repo_id="canrager/graphing_eval_results_0122", filename=file, repo_type="dataset")
    obj = json.load(open(local, "r"))
    # print(repo.to_dict())
    eval_result_metrics = obj["eval_result_metrics"]
    # print(f"eval_result_metrics: {eval_result_metrics}")
    saes[sae].update({"core": eval_result_metrics})
    # tqdm.write(f"{sae}: {eval_result_metrics}")

In [80]:
with open("saes.json", "w") as f:
    json.dump(saes, f, indent=4)

In [ ]:
repo

DatasetDict({
    train: Dataset({
        features: ['eval_type_id', 'eval_config', 'eval_id', 'datetime_epoch_millis', 'eval_result_metrics', 'eval_result_details', 'sae_bench_commit_hash', 'sae_lens_id', 'sae_lens_release_id', 'sae_lens_version', 'sae_cfg_dict'],
        num_rows: 1
    })
})